# 01 - 奖励模型 (Reward Modeling)

## 学习目标

本notebook深入讲解RLHF中奖励模型的原理与实现，包括：

1. **Bradley-Terry模型** - 成对比较的数学原理
2. **奖励模型架构** - 从头实现奖励模型
3. **偏好数据集** - 数据格式与处理
4. **训练与评估** - 完整训练流程
5. **KL散度正则化** - 防止过拟合

---

## 1. 理论基础

### 1.1 为什么需要奖励模型？

人类反馈是离线的、稀疏的，我们需要一个奖励模型来泛化到新的状态-动作对。

```
人类反馈 (x, y_w, y_l) → 奖励模型 r(x, y) → 用于强化学习
```

### 1.2 Bradley-Terry模型

**核心假设**: 如果y_w的奖励高于y_l，则人类更偏好y_w。

$$P(y_w > y_l | x) = \sigma(r(x, y_w) - r(x, y_l))$$

其中 $\sigma(z) = \frac{1}{1+e^{-z}}$ 是sigmoid函数。

### 1.3 损失函数推导

最大化对数似然：
$$\mathcal{L} = \sum \log P(y_w > y_l | x)$$

等价于最小化负对数似然：
$$L_{RM} = -\mathbb{E}[\log \sigma(r(x, y_w) - r(x, y_l))]$$

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Any
from dataclasses import dataclass

from reward_model import (
    RewardModelConfig,
    PairwiseRewardModel,
    PreferenceDataset
)

print("环境导入完成！")

## 2. Bradley-Terry模型详解

In [ ]:
def sigmoid(x: np.ndarray) -> np.ndarray:
    """Sigmoid函数实现。"""
    return 1 / (1 + np.exp(-x))

def bradley_terry_probability(reward_w: float, reward_l: float) -> float:
    """计算Bradley-Terry模型偏好概率。
    
    P(y_w > y_l) = sigmoid(r(x, y_w) - r(x, y_l))
    """
    return sigmoid(reward_w - reward_l)

# 测试不同奖励差值的概率
reward_diffs = np.linspace(-5, 5, 100)
probs = bradley_terry_probability(reward_diffs, 0)

plt.figure(figsize=(10, 5))
plt.plot(reward_diffs, probs, 'b-', linewidth=2, label='P(y_w > y_l)')
plt.axhline(y=0.5, color='r', linestyle='--', label='随机选择')
plt.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('奖励差值 r(y_w) - r(y_l)', fontsize=12)
plt.ylabel('偏好概率', fontsize=12)
plt.title('Bradley-Terry模型：奖励差值 vs 偏好概率', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"当奖励差值为+2时，偏好y_w的概率: {bradley_terry_probability(2, 0):.4f}")
print(f"当奖励差值为-2时，偏好y_w的概率: {bradley_terry_probability(0, 2):.4f}")

In [ ]:
def compute_btl_loss(reward_w: np.ndarray, reward_l: np.ndarray) -> float:
    """计算Bradley-Terry损失。"""
    # 交叉熵损失: -log(sigmoid(r_w - r_l))
    diff = reward_w - reward_l
    # 数值稳定的sigmoid交叉熵实现
    # -log(sigmoid(x)) = log(1 + exp(-x)) = softplus(-x)
    losses = np.log1p(np.exp(-diff))
    return losses.mean()

# 测试损失计算
rewards_w = np.array([2.0, 1.0, 0.5, -0.5])
rewards_l = np.array([1.0, 0.5, 0.0, -1.0])

loss = compute_btl_loss(rewards_w, rewards_l)
print(f"Bradley-Terry损失: {loss:.4f}")

# 验证：奖励差越大，损失越小
print("\n奖励差 vs 损失:")
for rw, rl in zip(rewards_w, rewards_l):
    single_loss = compute_btl_loss(np.array([rw]), np.array([rl]))
    print(f"  r_w={rw:5.2f}, r_l={rl:5.2f}, diff={rw-rl:5.2f} → loss={single_loss:.4f}")

## 3. 偏好数据集管理

In [ ]:
# 创建偏好数据集
dataset = PreferenceDataset()

# 添加高质量偏好样本
preferences = [
    # (提示, 优选回答, 拒绝回答)
    ("什么是机器学习？", 
     "机器学习是人工智能的一个分支，通过算法让计算机从数据中学习规律并做出预测。",
     "不知道"),
    
    ("如何学习Python？", 
     "建议从基础语法开始，通过实践项目逐步提高，同时阅读优秀的代码和文档。",
     "去网上搜"),
    
    ("解释深度学习", 
     "深度学习是机器学习的一种方法，使用多层神经网络自动提取数据的层次化特征。",
     "很复杂"),
    
    ("Python和Java的区别", 
     "Python语法简洁、开发效率高；Java性能更好、生态更成熟。选择取决于应用场景。",
     "Python更好"),
    
    ("什么是过拟合？", 
     "过拟合是指模型在训练数据上表现很好，但在新数据上泛化能力差的现象。",
     "不清楚"),
]

for prompt, chosen, rejected in preferences:
    dataset.add(prompt, chosen, rejected)

print(f"数据集大小: {len(dataset)}")
print(f"\n样本预览:")
for i, sample in enumerate(dataset.samples[:3], 1):
    print(f"\n样本 {i}:")
    print(f"  提示: {sample.prompt[:40]}...")
    print(f"  优选: {sample.chosen[:40]}...")
    print(f"  拒绝: {sample.rejected[:40]}...")

In [ ]:
# 批次采样
batch = dataset.sample(batch_size=4)

print(f"\n批次信息:")
print(f"批次大小: {len(batch.prompts)}")
print(f"提示数量: {len(batch.prompts)}")
print(f"优选回答数量: {len(batch.chosen_responses)}")
print(f"拒绝回答数量: {len(batch.rejected_responses)}")

## 4. 奖励模型训练

In [ ]:
# 创建奖励模型
config = RewardModelConfig(
    hidden_size=256,
    temperature=1.0,
    dropout=0.1
)
model = PairwiseRewardModel(config)

print(f"模型配置:")
print(f"  隐藏层大小: {config.hidden_size}")
print(f"  温度参数: {config.temperature}")
print(f"  Dropout率: {config.dropout}")

In [ ]:
# 训练循环
print("\n开始训练...")
print("="*60)

history = {'loss': [], 'accuracy': []}

for epoch in range(10):
    # 采样批次
    batch = dataset.sample(batch_size=len(dataset))
    
    # 训练步骤
    metrics = model.train_step(batch, learning_rate=1e-3)
    
    # 记录历史
    history['loss'].append(metrics['loss'])
    history['accuracy'].append(metrics['accuracy'])
    
    # 每2个epoch打印一次
    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch+1:2d}: "
              f"loss={metrics['loss']:.4f}, "
              f"accuracy={metrics['accuracy']:.2%}")

print("\n训练完成！")
print(f"最终准确率: {history['accuracy'][-1]:.2%}")

In [ ]:
# 可视化训练过程
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 损失曲线
axes[0].plot(history['loss'], 'b-o', linewidth=2, markersize=6)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('训练损失', fontsize=14)
axes[0].grid(True, alpha=0.3)

# 准确率曲线
axes[1].plot(history['accuracy'], 'g-o', linewidth=2, markersize=6)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('准确率', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 奖励模型评估

In [ ]:
# 测试奖励计算
test_cases = [
    {
        "prompt": "什么是神经网络？",
        "good": "神经网络是一种模拟人脑神经元结构的计算模型，由多层相互连接的节点组成。",
        "bad": "不清楚。"
    },
    {
        "prompt": "如何学习编程？",
        "good": "建议选择一门语言，从基础语法开始，通过做项目来实践，同时阅读优质代码。",
        "bad": "多练习。"
    },
    {
        "prompt": "解释什么是过拟合",
        "good": "过拟合是指模型在训练数据上表现很好，但在新数据上泛化能力差。",
        "bad": "不知道"
    },
]

print("奖励模型评估结果:")
print("="*70)

for i, case in enumerate(test_cases, 1):
    good_reward = model.compute_reward(case['prompt'], case['good'])
    bad_reward = model.compute_reward(case['prompt'], case['bad'])
    margin = good_reward - bad_reward
    
    print(f"\n测试 {i}:")
    print(f"提示: {case['prompt']}")
    print(f"好回答奖励: {good_reward:.4f}")
    print(f"差回答奖励: {bad_reward:.4f}")
    print(f"奖励差距: {margin:.4f}", end="")
    print(" ✓" if margin > 0 else " ✗")

## 6. KL散度正则化

防止奖励模型过拟合，保持与初始模型的接近。

In [ ]:
def compute_kl_divergence(p: np.ndarray, q: np.ndarray, eps: float = 1e-8) -> float:
    """计算两个分布的KL散度。
    
    KL(p||q) = Σ p(x) * log(p(x) / q(x))
    """
    p = np.clip(p, eps, 1)
    q = np.clip(q, eps, 1)
    return np.sum(p * np.log(p / q))

# 模拟奖励分布与初始分布
current_rewards = np.array([0.8, 0.6, 0.9, 0.7])
initial_rewards = np.array([0.5, 0.5, 0.5, 0.5])  # 初始均匀分布

# 转换为概率分布（通过softmax）
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

p = softmax(current_rewards)
q = softmax(initial_rewards)

kl = compute_kl_divergence(p, q)
print(f"KL散度(当前||初始): {kl:.4f}")
print(f"\n解释: KL散度越大，说明奖励模型偏离初始模型越远")

## 7. 批次编码与嵌入

In [ ]:
# 查看模型内部嵌入
batch = dataset.sample(2)

# 获取文本嵌入（简化版）
embeddings_chosen = model._encode_texts(batch.chosen_responses)
embeddings_rejected = model._encode_texts(batch.rejected_responses)

print(f"优选回答嵌入形状: {embeddings_chosen.shape}")
print(f"拒绝回答嵌入形状: {embeddings_rejected.shape}")
print(f"嵌入维度: {embeddings_chosen.shape[-1]}")

## 8. 奖励模型应用场景

In [ ]:
# 模拟多个候选回答的排序
prompt = "什么是机器学习？"

candidates = [
    "机器学习是让计算机从数据中学习的算法。",
    "不知道。",
    "ML就是机器学习，很复杂。",
    "机器学习通过算法让计算机从数据中提取规律并做出预测。",
    "去百度。",
]

# 计算每个候选的奖励
rewards = [model.compute_reward(prompt, c) for c in candidates]

# 排序
ranked = sorted(zip(candidates, rewards), key=lambda x: -x[1])

print(f"提示: {prompt}\n")
print("候选回答排名:\n")
for i, (text, reward) in enumerate(ranked, 1):
    print(f"{i}. [{reward:.4f}] {text}")

## 总结

本notebook介绍了奖励模型的核心内容：

### 关键要点

1. **Bradley-Terry模型**: P(y_w > y_l) = σ(r_w - r_l)
2. **损失函数**: L = -log σ(r_w - r_l)
3. **数据格式**: (prompt, chosen, rejected) 三元组
4. **KL正则化**: 防止过拟合，保持稳定性

### 下一步

- 学习PPO强化学习算法 (02_rlhf_training.ipynb)
- 了解DPO直接偏好优化 (03_dpo_training.ipynb)